# JEDI Ridge Score Stacking Workbook

**Objective:** predict profile-fit scores with Ridge, append predicted scores, and test whether Logistic Regression improves on `applied`.

**Input:** `data/jedi_master_prepared.csv`

**Outputs:** stacked model comparisons, test predictions, OOF predicted-score dataset.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
DATA_PATH = Path("data/jedi_master_prepared.csv")
OUTPUT_DIR = Path("data")


## Step 1: Load Prepared Dataset


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head(3))
display(df["applied"].value_counts().rename_axis("applied").reset_index(name="rows"))

Rows: 331
Columns: 51


,row_id,source_platform,source_type,scraped_at,can_apply_now,company_name,company_match_score,role_title,role_title_preference_score,role_family,role_family_preference_score,seniority,seniority_profile_fit_score,location,location_profile_fit_score,work_mode,work_mode_fit_score,employment_type,experience_min_years,experience_max_years,experience_profile_fit_score,skills_or_requirements,skills_profile_fit_score,salary_min,salary_max,salary_profile_fit_score,freshness,overall_preference_score,preference_reason,applied,apply_url,skills_or_requirements_original,skills_array,has_skills_or_requirements,skills_imputation_type,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,is_bengaluru,is_gurgaon_ncr,is_india,experience_min_bucket,salary_midpoint,salary_available,near_duplicate_key,company_name_standard,role_title_standard
0,1,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Nirmata,4,"Software Engineer Intern, India",5,Backend / Cloud,4,intern,5,"Bangalore, Hybrid",4,Hybrid,5,internship,0.0,1.0,5.0,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,2,1200000,1200000,4,Not visible,3,"Intern role is fine, everything is good but the requirements state Kubernetes, observability tools, promethus and Go which dont match wi...",no,https://job-boards.greenhouse.io/nirmata/jobs/5040355008,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,"[""go"",""java"",""microservices"",""kubernetes"",""prometheus"",""timescale"",""reliability"",""automation""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,1200000,True,nirmata|software_engineer_intern_india|bengaluru,nirmata,software_engineer_intern_india
1,2,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,CloudSEK,5,SDE - Backend - Intern,5,Backend,5,intern,5,"Bengaluru, Karnataka, India",4,Hybrid,5,internship,0.0,1.0,5.0,Backend engineering; cybersecurity product; APIs; databases likely,4,800000,900000,3,Not visible,4,"Backend intern in Bengaluru; very relevant for fresher profile, salary is a little less and location bengaluru",yes,https://job-boards.greenhouse.io/cloudsek/jobs/5766402004,Backend engineering; cybersecurity product; APIs; databases likely,"[""backend engineering"",""cybersecurity product"",""apis"",""databases likely""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,850000,True,cloudsek|sde_backend_intern|bengaluru,cloudsek,sde_backend_intern
2,3,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Anaplan,5,Associate Software Engineer,5,Backend / Fullstack,5,entry_level,5,"Gurugram, India",5,unknown,5,full_time,0.0,2.0,5.0,Software engineering; AI-infused platform; SaaS; cloud; planning systems,5,500000,700000,1,Not visible,3,Associate level in Gurgaon; high-quality company and location fitbut salary is quite less and the bad factor,yes,https://job-boards.greenhouse.io/anaplan/jobs/8511489002,Software engineering; AI-infused platform; SaaS; cloud; planning systems,"[""software engineering"",""ai-infused platform"",""saas"",""cloud"",""planning systems""]",True,original,fullstack,entry_level,full_time,unknown,delhi_ncr,india,delhi_ncr,False,True,True,0,600000,True,anaplan|associate_software_engineer|delhi_ncr,anaplan,associate_software_engineer


,applied,rows
0,no,205
1,yes,126


## Step 2: Define Targets


In [ ]:
SCORE_TARGETS = [
    "overall_preference_score",
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "skills_profile_fit_score",
]

missing_score_targets = [c for c in SCORE_TARGETS if c not in df.columns]
if missing_score_targets:
    raise ValueError(f"Missing score target columns: {missing_score_targets}")

df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
if not set(df["applied_clean"].unique()).issubset({"yes", "no"}):
    raise ValueError(f"Applied column has non-binary labels: {sorted(df['applied_clean'].unique())}")

df["target_applied"] = df["applied_clean"].map({"no": 0, "yes": 1}).astype(int)

score_audit = pd.DataFrame({
    "score_target": SCORE_TARGETS,
    "missing": [df[c].isna().sum() for c in SCORE_TARGETS],
    "mean": [df[c].mean() for c in SCORE_TARGETS],
    "std": [df[c].std() for c in SCORE_TARGETS],
    "min": [df[c].min() for c in SCORE_TARGETS],
    "max": [df[c].max() for c in SCORE_TARGETS],
})

display(score_audit)
display(df[["applied", "target_applied"]].value_counts().rename("rows").reset_index())

,score_target,missing,mean,std,min,max
0,overall_preference_score,0,3.099698,1.413903,0,5
1,role_title_preference_score,0,3.963746,1.430792,0,5
2,role_family_preference_score,0,4.441088,0.930300,1,5
3,seniority_profile_fit_score,0,3.486405,1.485984,0,5
4,location_profile_fit_score,0,3.199396,1.644863,0,5
5,skills_profile_fit_score,0,4.096677,1.019368,1,5


,applied,target_applied,rows
0,no,0,205
1,yes,1,126


## Step 3: Remove Leakage And Metadata


In [ ]:
ALL_SCORE_COLUMNS = [
    c for c in df.columns
    if c.endswith("_score") or c in ["company_match_score", "overall_preference_score"]
]

LEAKAGE_OR_METADATA_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_OR_METADATA_COLUMNS))
available_drop_columns = [c for c in drop_columns if c in df.columns]
candidate_raw_features = [c for c in df.columns if c not in available_drop_columns]

display(pd.DataFrame({"dropped_from_X": available_drop_columns}))
display(pd.DataFrame({"candidate_raw_feature": candidate_raw_features}))

,dropped_from_X
0,applied
1,applied_clean
2,apply_url
3,company_match_score
4,company_name_standard
5,experience_profile_fit_score
6,location_profile_fit_score
7,near_duplicate_key
8,overall_preference_score
9,preference_reason


,candidate_raw_feature
0,can_apply_now
1,company_name
2,role_title
3,role_family
4,seniority
5,location
6,work_mode
7,employment_type
8,experience_min_years
9,experience_max_years


## Step 4: Select Feature Groups


In [ ]:
text_source_columns = [
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "skills_or_requirements",
    "skills_array",
]

categorical_features = [
    "can_apply_now",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
    "freshness",
]

numeric_features = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
]

boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]

required_inputs = text_source_columns + categorical_features + numeric_features + boolean_features
missing_inputs = [c for c in required_inputs if c not in df.columns]
if missing_inputs:
    raise ValueError(f"Missing input columns: {missing_inputs}")

model_df = df.copy()
for col in text_source_columns:
    model_df[col] = model_df[col].fillna("").astype(str)
model_df["job_text"] = model_df[text_source_columns].agg(" ".join, axis=1)

for col in boolean_features:
    model_df[col] = model_df[col].fillna(False).astype(int)

base_feature_columns = ["job_text"] + categorical_features + numeric_features + boolean_features

valid_rows = model_df[SCORE_TARGETS].notna().all(axis=1) & model_df["target_applied"].notna()
X_raw = model_df.loc[valid_rows, base_feature_columns].copy()
y_scores = model_df.loc[valid_rows, SCORE_TARGETS].astype(float).copy()
y_applied = model_df.loc[valid_rows, "target_applied"].astype(int).copy()
meta = model_df.loc[valid_rows, [
    "row_id",
    "company_name",
    "role_title",
    "role_family_standard",
    "seniority_standard",
    "location_bucket",
    "work_mode_standard",
    "applied",
    "apply_url",
]].copy()

print(f"Rows used: {len(X_raw):,}")
display(X_raw.head(3))
display(y_scores.head(3))

Rows used: 331


,job_text,can_apply_now,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,experience_min_bucket,skills_imputation_type,freshness,experience_min_years,salary_min,salary_max,salary_midpoint,has_skills_or_requirements,is_bengaluru,is_gurgaon_ncr,is_india,salary_available
0,"Nirmata Software Engineer Intern, India Backend / Cloud intern Bangalore, Hybrid Go; Java; microservices; Kubernetes; Prometheus; TimeSc...",Yes,backend,intern,internship,hybrid,bengaluru,india,bengaluru,0,original,Not visible,0.0,1200000,1200000,1200000,1,1,0,1,1
1,"CloudSEK SDE - Backend - Intern Backend intern Bengaluru, Karnataka, India Backend engineering; cybersecurity product; APIs; databases l...",Yes,backend,intern,internship,hybrid,bengaluru,india,bengaluru,0,original,Not visible,0.0,800000,900000,850000,1,1,0,1,1
2,"Anaplan Associate Software Engineer Backend / Fullstack entry_level Gurugram, India Software engineering; AI-infused platform; SaaS; clo...",Yes,fullstack,entry_level,full_time,unknown,delhi_ncr,india,delhi_ncr,0,original,Not visible,0.0,500000,700000,600000,1,0,1,1,1


,overall_preference_score,role_title_preference_score,role_family_preference_score,seniority_profile_fit_score,location_profile_fit_score,skills_profile_fit_score
0,3.0,5.0,4.0,5.0,4.0,2.0
1,4.0,5.0,5.0,5.0,4.0,4.0
2,3.0,5.0,5.0,5.0,5.0,5.0


## Step 5: Build Pipelines


In [ ]:
def make_preprocess(extra_numeric_features=None, text_max_features=900):
    extra_numeric_features = list(extra_numeric_features or [])
    numeric_cols = numeric_features + boolean_features + extra_numeric_features

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
    ])

    text_transformer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=text_max_features,
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_cols),
            ("categorical", categorical_transformer, categorical_features),
            ("text", text_transformer, "job_text"),
        ]
    )


def make_score_model():
    return Pipeline(steps=[
        ("preprocess", make_preprocess(text_max_features=1200)),
        ("regressor", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ])


def make_logistic_model(extra_numeric_features=None):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(extra_numeric_features=extra_numeric_features, text_max_features=800)),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
            C=1.0,
            random_state=RANDOM_STATE,
        )),
    ])


def clip_scores(predictions):
    return np.clip(np.asarray(predictions, dtype=float), 0.0, 5.0)


PRED_SCORE_COLUMNS = [f"pred_{c}" for c in SCORE_TARGETS]
TRUE_SCORE_COLUMNS = [f"true_{c}" for c in SCORE_TARGETS]


def append_score_columns(X_base, score_array, prefix):
    X_aug = X_base.copy()
    score_array = clip_scores(score_array)
    for idx, score_name in enumerate(SCORE_TARGETS):
        X_aug[f"{prefix}_{score_name}"] = score_array[:, idx]
    return X_aug

## Step 6: Define Evaluation Helpers


In [ ]:
def score_regression_metrics(y_true, y_pred, split_name):
    y_true_df = pd.DataFrame(np.asarray(y_true, dtype=float), columns=SCORE_TARGETS)
    y_pred_df = pd.DataFrame(clip_scores(y_pred), columns=SCORE_TARGETS)
    rows = []
    for target in SCORE_TARGETS:
        actual = y_true_df[target]
        pred = y_pred_df[target]
        abs_error = (pred - actual).abs()
        rows.append({
            "split": split_name,
            "target": target,
            "mae": abs_error.mean(),
            "rmse": np.sqrt(np.mean(np.square(pred - actual))),
            "r2": r2_score(actual, pred),
            "spearman": actual.corr(pred, method="spearman"),
            "within_0_5": (abs_error <= 0.5).mean(),
            "within_1_0": (abs_error <= 1.0).mean(),
        })
    return pd.DataFrame(rows)


def classification_summary(model_name, split_name, y_true, proba, threshold=0.50):
    pred = (np.asarray(proba) >= threshold).astype(int)
    return {
        "model": model_name,
        "split": split_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_yes": precision_score(y_true, pred, zero_division=0),
        "recall_yes": recall_score(y_true, pred, zero_division=0),
        "f1_yes": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "average_precision": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba),
        "brier_score": brier_score_loss(y_true, proba),
        "predicted_yes_count": int(pred.sum()),
    }


def threshold_table(y_true, proba, model_name):
    rows = []
    for threshold in np.arange(0.20, 0.81, 0.05):
        pred = (np.asarray(proba) >= threshold).astype(int)
        rows.append({
            "model": model_name,
            "threshold": round(float(threshold), 2),
            "predicted_yes_count": int(pred.sum()),
            "precision_yes": precision_score(y_true, pred, zero_division=0),
            "recall_yes": recall_score(y_true, pred, zero_division=0),
            "f1_yes": f1_score(y_true, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)

## Step 7: Create Holdout Split


In [ ]:
train_pos, test_pos = train_test_split(
    np.arange(len(X_raw)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_applied,
)

X_train_raw = X_raw.iloc[train_pos].copy()
X_test_raw = X_raw.iloc[test_pos].copy()
y_train_scores = y_scores.iloc[train_pos].copy()
y_test_scores = y_scores.iloc[test_pos].copy()
y_train = y_applied.iloc[train_pos].copy()
y_test = y_applied.iloc[test_pos].copy()
meta_train = meta.iloc[train_pos].copy()
meta_test = meta.iloc[test_pos].copy()

print(f"Train rows: {len(X_train_raw):,}")
print(f"Test rows:  {len(X_test_raw):,}")
display(pd.DataFrame({
    "set": ["train", "train", "test", "test"],
    "class": [0, 1, 0, 1],
    "count": [int((y_train == 0).sum()), int((y_train == 1).sum()), int((y_test == 0).sum()), int((y_test == 1).sum())],
}))

Train rows: 264
Test rows:  67


,set,class,count
0,train,0,164
1,train,1,100
2,test,0,41
3,test,1,26


## Step 8: Predict Profile Scores With Ridge


In [ ]:
score_inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
score_model = make_score_model()

train_score_oof = cross_val_predict(
    score_model,
    X_train_raw,
    y_train_scores,
    cv=score_inner_cv,
    n_jobs=None,
)
train_score_oof = clip_scores(train_score_oof)

score_model.fit(X_train_raw, y_train_scores)
test_score_pred = clip_scores(score_model.predict(X_test_raw))

X_train_stacked = append_score_columns(X_train_raw, train_score_oof, prefix="pred")
X_test_stacked = append_score_columns(X_test_raw, test_score_pred, prefix="pred")

X_train_oracle = append_score_columns(X_train_raw, y_train_scores.values, prefix="true")
X_test_oracle = append_score_columns(X_test_raw, y_test_scores.values, prefix="true")

score_train_metrics = score_regression_metrics(y_train_scores, train_score_oof, "train_oof")
score_test_metrics = score_regression_metrics(y_test_scores, test_score_pred, "holdout_test")

display(pd.concat([score_train_metrics, score_test_metrics], ignore_index=True))
display(X_train_stacked[PRED_SCORE_COLUMNS].head())

,split,target,mae,rmse,r2,spearman,within_0_5,within_1_0
0,train_oof,overall_preference_score,0.571487,0.749674,0.718216,0.843505,0.530303,0.852273
1,train_oof,role_title_preference_score,0.622932,0.871068,0.647975,0.743164,0.481061,0.784091
2,train_oof,role_family_preference_score,0.442293,0.702442,0.463029,0.659086,0.681818,0.871212
3,train_oof,seniority_profile_fit_score,0.403126,0.649181,0.814886,0.915844,0.757576,0.878788
4,train_oof,location_profile_fit_score,0.492418,0.690044,0.827884,0.891408,0.640152,0.886364
5,train_oof,skills_profile_fit_score,0.500542,0.742171,0.498402,0.678675,0.632576,0.837121
6,holdout_test,overall_preference_score,0.434977,0.571830,0.829873,0.908295,0.656716,0.895522
7,holdout_test,role_title_preference_score,0.723057,1.032404,0.287152,0.569540,0.477612,0.776119
8,holdout_test,role_family_preference_score,0.460450,0.632269,0.315061,0.577366,0.626866,0.850746
9,holdout_test,seniority_profile_fit_score,0.333356,0.451445,0.890566,0.937836,0.761194,0.955224


,pred_overall_preference_score,pred_role_title_preference_score,pred_role_family_preference_score,pred_seniority_profile_fit_score,pred_location_profile_fit_score,pred_skills_profile_fit_score
175,2.543312,2.747429,4.434349,1.300347,4.161550,4.152296
79,3.485828,3.049030,4.412391,2.683498,4.892154,4.820515
283,2.495982,2.958942,5.000000,2.441142,3.422503,3.462100
262,3.346238,4.717466,3.667473,5.000000,0.605060,3.761477
149,2.570308,5.000000,4.466167,4.004238,1.666039,4.914197


## Step 9: Train Logistic Variants


In [ ]:
holdout_configs = [
    ("raw_logistic", make_logistic_model(), X_train_raw, X_test_raw, []),
    ("stacked_predicted_scores_logistic", make_logistic_model(PRED_SCORE_COLUMNS), X_train_stacked, X_test_stacked, PRED_SCORE_COLUMNS),
    ("oracle_true_scores_logistic", make_logistic_model(TRUE_SCORE_COLUMNS), X_train_oracle, X_test_oracle, TRUE_SCORE_COLUMNS),
]

holdout_summaries = []
holdout_probabilities = {}
holdout_models = {}

for model_name, estimator, X_tr, X_te, extra_cols in holdout_configs:
    estimator.fit(X_tr, y_train)
    proba = estimator.predict_proba(X_te)[:, 1]
    holdout_probabilities[model_name] = proba
    holdout_models[model_name] = estimator
    holdout_summaries.append(classification_summary(model_name, "holdout_test", y_test, proba, threshold=0.50))

holdout_summary_df = pd.DataFrame(holdout_summaries).sort_values("f1_yes", ascending=False).reset_index(drop=True)
display(holdout_summary_df)

,model,split,threshold,accuracy,balanced_accuracy,precision_yes,recall_yes,f1_yes,roc_auc,average_precision,log_loss,brier_score,predicted_yes_count
0,stacked_predicted_scores_logistic,holdout_test,0.5,0.910448,0.905722,0.884615,0.884615,0.884615,0.941839,0.928013,0.300182,0.085422,26
1,oracle_true_scores_logistic,holdout_test,0.5,0.895522,0.900563,0.827586,0.923077,0.872727,0.953096,0.935573,0.296229,0.087022,29
2,raw_logistic,holdout_test,0.5,0.880597,0.874296,0.846154,0.846154,0.846154,0.932458,0.907225,0.315456,0.089136,26


## Step 10: Inspect Chosen Model


In [ ]:
chosen_model_name = "stacked_predicted_scores_logistic"
chosen_proba = holdout_probabilities[chosen_model_name]
chosen_pred = (chosen_proba >= 0.50).astype(int)

print(classification_report(y_test, chosen_pred, target_names=["no", "yes"], zero_division=0))
display(pd.DataFrame(
    confusion_matrix(y_test, chosen_pred),
    index=["actual_no", "actual_yes"],
    columns=["pred_no", "pred_yes"],
))
display(threshold_table(y_test, chosen_proba, chosen_model_name))

              precision    recall  f1-score   support

          no       0.93      0.93      0.93        41
         yes       0.88      0.88      0.88        26

    accuracy                           0.91        67
   macro avg       0.91      0.91      0.91        67
weighted avg       0.91      0.91      0.91        67



,pred_no,pred_yes
actual_no,38,3
actual_yes,3,23


,model,threshold,predicted_yes_count,precision_yes,recall_yes,f1_yes,balanced_accuracy
0,stacked_predicted_scores_logistic,0.20,37,0.675676,0.961538,0.793651,0.834428
1,stacked_predicted_scores_logistic,0.25,34,0.705882,0.923077,0.800000,0.839587
2,stacked_predicted_scores_logistic,0.30,31,0.774194,0.923077,0.842105,0.876173
3,stacked_predicted_scores_logistic,0.35,29,0.793103,0.884615,0.836364,0.869137
4,stacked_predicted_scores_logistic,0.40,27,0.851852,0.884615,0.867925,0.893527
5,stacked_predicted_scores_logistic,0.45,26,0.884615,0.884615,0.884615,0.905722
6,stacked_predicted_scores_logistic,0.50,26,0.884615,0.884615,0.884615,0.905722
7,stacked_predicted_scores_logistic,0.55,25,0.880000,0.846154,0.862745,0.886492
8,stacked_predicted_scores_logistic,0.60,25,0.880000,0.846154,0.862745,0.886492
9,stacked_predicted_scores_logistic,0.65,24,0.875000,0.807692,0.840000,0.867261


## Step 11: Compare Holdout Predictions


In [ ]:
prediction_review = meta_test.reset_index(drop=True).copy()
prediction_review["actual_applied"] = y_test.reset_index(drop=True).values

for model_name, proba in holdout_probabilities.items():
    prediction_review[f"{model_name}_probability"] = proba
    prediction_review[f"{model_name}_prediction"] = (proba >= 0.50).astype(int)

for idx, score_name in enumerate(SCORE_TARGETS):
    prediction_review[f"actual_{score_name}"] = y_test_scores.reset_index(drop=True)[score_name]
    prediction_review[f"pred_{score_name}"] = test_score_pred[:, idx]

prediction_review["stacked_minus_raw_probability"] = (
    prediction_review["stacked_predicted_scores_logistic_probability"]
    - prediction_review["raw_logistic_probability"]
)

display(prediction_review.sort_values("stacked_predicted_scores_logistic_probability", ascending=False).head(20))
display(prediction_review.reindex(prediction_review["stacked_minus_raw_probability"].abs().sort_values(ascending=False).index).head(20))

,row_id,company_name,role_title,role_family_standard,seniority_standard,location_bucket,work_mode_standard,applied,apply_url,actual_applied,raw_logistic_probability,raw_logistic_prediction,stacked_predicted_scores_logistic_probability,stacked_predicted_scores_logistic_prediction,oracle_true_scores_logistic_probability,oracle_true_scores_logistic_prediction,actual_overall_preference_score,pred_overall_preference_score,actual_role_title_preference_score,pred_role_title_preference_score,actual_role_family_preference_score,pred_role_family_preference_score,actual_seniority_profile_fit_score,pred_seniority_profile_fit_score,actual_location_profile_fit_score,pred_location_profile_fit_score,actual_skills_profile_fit_score,pred_skills_profile_fit_score,stacked_minus_raw_probability
59,5,Energy Exemplar,Associate Software Engineer - Python,backend,entry_level,bengaluru,unknown,yes,https://job-boards.greenhouse.io/energyexemplarllc/jobs/5190824008,1,0.991751,1,0.994466,1,0.999819,1,5.0,4.280692,5.0,4.513321,5.0,4.155347,5.0,4.991171,4.0,4.761553,5.0,4.007430,0.002715
58,203,Patronus,Full stack Engineer (AI & Data Pipelines),fullstack,junior,bengaluru,onsite,yes,https://wellfound.com/role/l/frontend-engineer/india,1,0.991102,1,0.993199,1,0.996846,1,5.0,5.000000,5.0,5.000000,5.0,5.000000,5.0,5.000000,4.0,3.983121,5.0,5.000000,0.002096
38,210,ParallelDots,Machine Learning Engineer,ml_ai,intern,india_remote,remote,yes,https://wellfound.com/role/l/machine-learning-engineer/india,1,0.984426,1,0.991171,1,0.997120,1,5.0,4.746553,4.0,4.608776,4.0,4.229525,5.0,5.000000,5.0,4.806262,4.0,4.326507,0.006745
9,255,Weekday,Backend Engineer,backend,entry_level,bengaluru,onsite,yes,https://jobs.lever.co/weekdayworks,1,0.981966,1,0.989544,1,0.990170,1,5.0,4.946364,5.0,5.000000,5.0,5.000000,5.0,4.829098,4.0,4.155506,3.0,4.123505,0.007578
52,287,TopGrep Tech,Software Engineer Intern to hire,fullstack,intern,bengaluru,onsite,yes,https://wellfound.com/company/topgrep-tech/jobs,1,0.986734,1,0.988501,1,0.975304,1,4.0,4.224134,5.0,5.000000,5.0,4.829207,5.0,5.000000,4.0,3.603028,4.0,4.358034,0.001768
23,173,360customizer,Backend Engineer,backend,junior,pune,onsite,yes,https://wellfound.com/role/l/backend-engineer/india,1,0.987101,1,0.986497,1,0.982209,1,4.0,4.197378,5.0,4.552916,5.0,4.521424,5.0,5.000000,3.0,3.372240,4.0,4.115487,-0.000604
61,309,Lokal App,Backend SDE-1,backend,entry_level,bengaluru,onsite,yes,https://jobs.workable.com/view/cTvws24CsRBbgbok7QgCGo/backend-sde--1-in-bengaluru-at-lokal-app,1,0.983699,1,0.984406,1,0.997264,1,5.0,4.496421,5.0,4.666164,5.0,4.701233,5.0,4.664841,4.0,4.311487,4.0,3.664607,0.000706
31,161,Rankscope,Full Stack Engineer,fullstack,junior,global_remote,remote,yes,https://wellfound.com/role/l/full-stack-engineer/india,1,0.969285,1,0.983527,1,0.994303,1,5.0,4.830476,5.0,5.000000,5.0,4.952263,5.0,4.838677,4.0,3.585703,5.0,5.000000,0.014243
50,290,TrueBuilt Software,Full-Stack Engineer,fullstack,junior,india_remote,remote,yes,https://wellfound.com/jobs/4395820-full-stack-engineer,1,0.964969,1,0.979708,1,0.997622,1,5.0,3.974411,5.0,4.638492,5.0,5.000000,5.0,3.973272,4.0,5.000000,5.0,4.157035,0.014739
34,328,Naukri Search Seed - Gurgaon,Backend Developer 0-2 Years,backend,entry_level,delhi_ncr,onsite,yes,https://www.naukri.com/backend-developer-0-to-2-years-jobs-in-gurgaon,1,0.962929,1,0.977144,1,0.991202,1,5.0,4.936348,5.0,4.901891,5.0,5.000000,5.0,5.000000,5.0,5.000000,4.0,3.518277,0.014215


,row_id,company_name,role_title,role_family_standard,seniority_standard,location_bucket,work_mode_standard,applied,apply_url,actual_applied,raw_logistic_probability,raw_logistic_prediction,stacked_predicted_scores_logistic_probability,stacked_predicted_scores_logistic_prediction,oracle_true_scores_logistic_probability,oracle_true_scores_logistic_prediction,actual_overall_preference_score,pred_overall_preference_score,actual_role_title_preference_score,pred_role_title_preference_score,actual_role_family_preference_score,pred_role_family_preference_score,actual_seniority_profile_fit_score,pred_seniority_profile_fit_score,actual_location_profile_fit_score,pred_location_profile_fit_score,actual_skills_profile_fit_score,pred_skills_profile_fit_score,stacked_minus_raw_probability
44,232,"Clerky, Inc.",Customer Operations & Writing Specialist,other,unknown,global_remote,remote,no,https://remotive.com/remote-jobs/customer-service/customer-operations-writing-specialist-2090942,0,0.537838,1,0.352926,0,0.012678,0,2.0,3.011068,1.0,3.259405,1.0,1.714700,4.0,3.697357,4.0,3.691233,1.0,3.280688,-0.184913
13,51,Netomi,SDE I (FullStack),fullstack,entry_level,india_remote,remote,yes,https://jobs.lever.co/netomi/5dfe7149-e3e7-4063-be6f-440081b188b5,1,0.452793,0,0.548734,1,0.869896,1,5.0,4.014828,5.0,4.370323,5.0,4.879695,5.0,4.687040,5.0,4.999221,5.0,4.731394,0.095942
19,33,Fluxon,Senior Software Engineer (AI Engineer),backend,senior,india_remote,remote,no,https://job-boards.greenhouse.io/fluxon/jobs/4648524005,0,0.250479,0,0.171156,0,0.007948,0,0.0,1.649111,2.0,2.575214,4.0,3.945574,2.0,1.270755,5.0,4.343577,4.0,3.117305,-0.079323
25,127,Anovium,Senior Full Stack Javascript Node Backend Engineer [REMOTE],fullstack,senior,global_remote,remote,no,https://jobs.lever.co/Anovium/c55c1aa9-2bb0-4447-bd40-cdc616adb9c2,0,0.126016,0,0.185219,0,0.105942,0,3.0,2.969856,2.0,3.097200,5.0,5.000000,2.0,2.224561,4.0,3.756808,5.0,4.993955,0.059203
16,318,Funding Societies | Modalku Group,Product Security Intern,other,intern,india_remote,remote,yes,https://jobs.workable.com/view/vShUDs56ULYd65KJHZV74z/remote-product-security-intern-in-india-at-funding-societies-%7C-modalku-group,1,0.936149,1,0.883558,1,0.915700,1,4.0,3.904515,4.0,3.075801,4.0,2.692647,5.0,4.768417,4.0,4.528603,3.0,3.441043,-0.052591
24,163,MaxHome.ai,Full-stack engineer,fullstack,senior,bengaluru,onsite,no,https://wellfound.com/role/l/full-stack-engineer/india,0,0.317077,0,0.368838,0,0.136585,0,3.0,2.995406,5.0,3.980897,5.0,4.895591,1.0,1.843504,4.0,3.840839,5.0,5.000000,0.051761
62,117,Included Health,Data Scientist II,data_science,unknown,unknown,remote,no,https://jobs.lever.co/includedhealth/785d36f2-f2e2-4c56-8c4b-ec80d86adc3a,0,0.230907,0,0.281081,0,0.566757,1,4.0,3.342991,3.0,4.522743,4.0,4.102992,3.0,3.936979,3.0,2.265699,5.0,4.720299,0.050174
42,308,WealthKernel / Similar AI SaaS,Senior Software Development Engineer (Payments),backend,senior,india_remote,remote,no,https://wellfound.com/jobs/4201589-senior-software-development-engineer-payments,0,0.329259,0,0.279914,0,0.176195,0,2.0,1.975151,5.0,2.466003,5.0,5.000000,2.0,1.279114,4.0,4.703728,5.0,4.161544,-0.049345
54,88,Bolster,"Senior Software Engineer, Backend - India",backend,senior,india_remote,remote,no,https://jobs.lever.co/bolster/cd85d0f8-51c4-467f-9e4d-cd04b2a3be47,0,0.179978,0,0.140243,0,0.066324,0,3.0,3.209853,2.0,1.797452,5.0,4.819969,2.0,2.304300,5.0,5.000000,4.0,3.970677,-0.039735
48,162,Kongu Engineering and Software Systems,Full-Stack Engineer,fullstack,early_mid,india_remote,remote,no,https://wellfound.com/role/l/full-stack-engineer/india,0,0.392761,0,0.432200,0,0.431963,0,4.0,3.977131,5.0,4.805497,5.0,4.860253,3.0,3.134397,5.0,4.935769,5.0,5.000000,0.039440


## Step 12: Run Leakage-Safe Outer CV


In [ ]:
def run_outer_cv_comparison():
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X_raw, y_applied), start=1):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_tr = y_applied.iloc[train_idx].copy()
        y_val = y_applied.iloc[val_idx].copy()
        y_tr_scores = y_scores.iloc[train_idx].copy()
        y_val_scores = y_scores.iloc[val_idx].copy()

        fold_score_model = make_score_model()
        tr_score_oof = clip_scores(cross_val_predict(fold_score_model, X_tr_raw, y_tr_scores, cv=inner_cv, n_jobs=None))
        fold_score_model.fit(X_tr_raw, y_tr_scores)
        val_score_pred = clip_scores(fold_score_model.predict(X_val_raw))

        X_tr_stacked = append_score_columns(X_tr_raw, tr_score_oof, prefix="pred")
        X_val_stacked = append_score_columns(X_val_raw, val_score_pred, prefix="pred")
        X_tr_oracle = append_score_columns(X_tr_raw, y_tr_scores.values, prefix="true")
        X_val_oracle = append_score_columns(X_val_raw, y_val_scores.values, prefix="true")

        fold_configs = [
            ("raw_logistic", make_logistic_model(), X_tr_raw, X_val_raw),
            ("stacked_predicted_scores_logistic", make_logistic_model(PRED_SCORE_COLUMNS), X_tr_stacked, X_val_stacked),
            ("oracle_true_scores_logistic", make_logistic_model(TRUE_SCORE_COLUMNS), X_tr_oracle, X_val_oracle),
        ]

        for model_name, estimator, X_tr, X_val in fold_configs:
            estimator.fit(X_tr, y_tr)
            proba = estimator.predict_proba(X_val)[:, 1]
            summary = classification_summary(model_name, f"outer_fold_{fold_idx}", y_val, proba, threshold=0.50)
            summary["fold"] = fold_idx
            rows.append(summary)

    return pd.DataFrame(rows)


outer_cv_results = run_outer_cv_comparison()
display(outer_cv_results)

,model,split,threshold,accuracy,balanced_accuracy,precision_yes,recall_yes,f1_yes,roc_auc,average_precision,log_loss,brier_score,predicted_yes_count,fold
0,raw_logistic,outer_fold_1,0.5,0.880597,0.860225,0.909091,0.769231,0.833333,0.945591,0.883219,0.326186,0.093729,22,1
1,stacked_predicted_scores_logistic,outer_fold_1,0.5,0.865672,0.840994,0.904762,0.730769,0.808511,0.942777,0.881978,0.323464,0.093095,21,1
2,oracle_true_scores_logistic,outer_fold_1,0.5,0.880597,0.874296,0.846154,0.846154,0.846154,0.949343,0.929458,0.284891,0.085877,26,1
3,raw_logistic,outer_fold_2,0.5,0.848485,0.846829,0.777778,0.840000,0.807692,0.895610,0.900624,0.382286,0.111954,27,2
4,stacked_predicted_scores_logistic,outer_fold_2,0.5,0.848485,0.854634,0.758621,0.880000,0.814815,0.913171,0.917679,0.354241,0.103927,29,2
5,oracle_true_scores_logistic,outer_fold_2,0.5,0.878788,0.863415,0.869565,0.800000,0.833333,0.959024,0.950335,0.242506,0.075998,23,2
6,raw_logistic,outer_fold_3,0.5,0.924242,0.923415,0.884615,0.920000,0.901961,0.979512,0.974988,0.231416,0.064086,26,3
7,stacked_predicted_scores_logistic,outer_fold_3,0.5,0.954545,0.955610,0.923077,0.960000,0.941176,0.986341,0.983034,0.220702,0.059430,26,3
8,oracle_true_scores_logistic,outer_fold_3,0.5,0.939394,0.943415,0.888889,0.960000,0.923077,0.987317,0.983828,0.170527,0.046780,27,3
9,raw_logistic,outer_fold_4,0.5,0.878788,0.863415,0.869565,0.800000,0.833333,0.904390,0.886560,0.358074,0.100967,23,4


## Step 13: Aggregate CV Results


In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_yes",
    "recall_yes",
    "f1_yes",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "predicted_yes_count",
]

outer_cv_summary = (
    outer_cv_results
    .groupby("model")[metric_columns]
    .agg(["mean", "std"])
)

display(outer_cv_summary)

flat_outer_cv_summary = outer_cv_summary.copy()
flat_outer_cv_summary.columns = [f"{metric}_{stat}" for metric, stat in flat_outer_cv_summary.columns]
flat_outer_cv_summary = flat_outer_cv_summary.reset_index().sort_values("f1_yes_mean", ascending=False)
display(flat_outer_cv_summary)

accuracy           balanced_accuracy  \
                                       mean       std              mean   
model                                                                     
oracle_true_scores_logistic        0.894301  0.025993          0.887152   
raw_logistic                       0.882180  0.027065          0.871460   
stacked_predicted_scores_logistic  0.888286  0.043377          0.879614   

                                            precision_yes            \
                                        std          mean       std   
model                                                                 
oracle_true_scores_logistic        0.032257      0.863922  0.020408   
raw_logistic                       0.029840      0.862123  0.049842   
stacked_predicted_scores_logistic  0.047310      0.866567  0.069945   

                                  recall_yes              f1_yes            \
                                        mean       std      mean       std   
model                                                                        
oracle_true_scores_logistic         0.857231  0.060319  0.859941  0.036365   
raw_logistic                        0.825846  0.058324  0.841931  0.035347   
stacked_predicted_scores_logistic   0.842154  0.085890  0.851166  0.057063   

                                    roc_auc           average_precision  \
                                       mean       std              mean   
model                                                                     
oracle_true_scores_logistic        0.958844  0.016480          0.941955   
raw_logistic                       0.929411  0.033897          0.904322   
stacked_predicted_scores_logistic  0.935872  0.031657          0.910415   

                                             log_loss           brier_score  \
                                        std      mean       std        mean   
model                                                                         
oracle_true_scores_logistic        0.027605  0.253150  0.051147    0.077428   
raw_logistic                       0.040490  0.331475  0.059392    0.094938   
stacked_predicted_scores_logistic  0.043330  0.325236  0.061174    0.092596   

                                            predicted_yes_count            
                                        std                mean       std  
model                                                                      
oracle_true_scores_logistic        0.018000                25.0  1.581139  
raw_logistic                       0.018443                24.2  2.167948  
stacked_predicted_scores_logistic  0.019085                24.6  3.049590

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_yes_mean,precision_yes_std,recall_yes_mean,recall_yes_std,f1_yes_mean,f1_yes_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,log_loss_mean,log_loss_std,brier_score_mean,brier_score_std,predicted_yes_count_mean,predicted_yes_count_std
0,oracle_true_scores_logistic,0.894301,0.025993,0.887152,0.032257,0.863922,0.020408,0.857231,0.060319,0.859941,0.036365,0.958844,0.016480,0.941955,0.027605,0.253150,0.051147,0.077428,0.018000,25.0,1.581139
2,stacked_predicted_scores_logistic,0.888286,0.043377,0.879614,0.047310,0.866567,0.069945,0.842154,0.085890,0.851166,0.057063,0.935872,0.031657,0.910415,0.043330,0.325236,0.061174,0.092596,0.019085,24.6,3.049590
1,raw_logistic,0.882180,0.027065,0.871460,0.029840,0.862123,0.049842,0.825846,0.058324,0.841931,0.035347,0.929411,0.033897,0.904322,0.040490,0.331475,0.059392,0.094938,0.018443,24.2,2.167948


## Step 14: Build OOF Predicted-Score Dataset


In [ ]:
full_score_oof = clip_scores(cross_val_predict(make_score_model(), X_raw, y_scores, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), n_jobs=None))

stacked_dataset = model_df.loc[valid_rows].copy()
for idx, score_name in enumerate(SCORE_TARGETS):
    stacked_dataset[f"pred_{score_name}"] = full_score_oof[:, idx]

display(stacked_dataset[["row_id", "company_name", "role_title", "applied"] + PRED_SCORE_COLUMNS + SCORE_TARGETS].head(10))

,row_id,company_name,role_title,applied,pred_overall_preference_score,pred_role_title_preference_score,pred_role_family_preference_score,pred_seniority_profile_fit_score,pred_location_profile_fit_score,pred_skills_profile_fit_score,overall_preference_score,role_title_preference_score,role_family_preference_score,seniority_profile_fit_score,location_profile_fit_score,skills_profile_fit_score
0,1,Nirmata,"Software Engineer Intern, India",no,4.020263,4.589475,4.225490,5.000000,4.045725,2.754153,3,5,4,5,4,2
1,2,CloudSEK,SDE - Backend - Intern,yes,3.821272,4.388506,4.280609,4.818390,3.938882,3.200478,4,5,5,5,4,4
2,3,Anaplan,Associate Software Engineer,yes,4.189437,3.906975,4.435274,5.000000,5.000000,4.123230,3,5,5,5,5,5
3,4,Redwood Software,Associate Software Engineer,no,2.542846,4.920001,4.095842,4.626963,3.827417,2.823582,3,5,5,5,3,3
4,5,Energy Exemplar,Associate Software Engineer - Python,yes,4.291964,4.883000,4.615432,5.000000,4.764803,4.133107,5,5,5,5,4,5
5,6,Conga,"Associate Software Engineer, TS",yes,3.777001,4.855960,5.000000,5.000000,4.129423,5.000000,5,5,5,5,5,5
6,7,Precision Medicine Group,Software Engineer I - Data,yes,4.054829,4.166451,3.854029,5.000000,4.638167,3.260931,5,5,3,5,5,4
7,8,DoorDash India,"Software Engineer, Backend",yes,2.267523,3.458061,4.419487,1.945927,3.423234,2.958601,4,5,5,3,4,4
8,9,Perch Energy,Software Engineer II,yes,2.858626,3.989286,4.944093,4.092748,3.749649,3.700504,3,3,5,4,3,5
9,10,Karat,Software Engineer (SWE3) (India),yes,1.891523,4.283855,4.290478,2.045006,4.482682,3.896085,2,3,5,3,4,4


## Step 15: Save Artifacts


In [ ]:
holdout_summary_path = OUTPUT_DIR / "jedi_stacked_logistic_holdout_comparison.csv"
outer_cv_detail_path = OUTPUT_DIR / "jedi_stacked_logistic_outer_cv_detail.csv"
outer_cv_summary_path = OUTPUT_DIR / "jedi_stacked_logistic_outer_cv_summary.csv"
prediction_review_path = OUTPUT_DIR / "jedi_stacked_logistic_test_predictions.csv"
stacked_dataset_path = OUTPUT_DIR / "jedi_master_with_oof_predicted_profile_scores.csv"
score_metrics_path = OUTPUT_DIR / "jedi_stacked_ridge_score_metrics.csv"

holdout_summary_df.to_csv(holdout_summary_path, index=False)
outer_cv_results.to_csv(outer_cv_detail_path, index=False)
flat_outer_cv_summary.to_csv(outer_cv_summary_path, index=False)
prediction_review.to_csv(prediction_review_path, index=False)
stacked_dataset.to_csv(stacked_dataset_path, index=False)
pd.concat([score_train_metrics, score_test_metrics], ignore_index=True).to_csv(score_metrics_path, index=False)

print(f"Saved: {holdout_summary_path}")
print(f"Saved: {outer_cv_detail_path}")
print(f"Saved: {outer_cv_summary_path}")
print(f"Saved: {prediction_review_path}")
print(f"Saved: {stacked_dataset_path}")
print(f"Saved: {score_metrics_path}")

Saved: data\jedi_stacked_logistic_holdout_comparison.csv
Saved: data\jedi_stacked_logistic_outer_cv_detail.csv
Saved: data\jedi_stacked_logistic_outer_cv_summary.csv
Saved: data\jedi_stacked_logistic_test_predictions.csv
Saved: data\jedi_master_with_oof_predicted_profile_scores.csv
Saved: data\jedi_stacked_ridge_score_metrics.csv


## Step 16: Final Decision Check

- Compare raw Logistic, stacked Logistic, and oracle-score Logistic.
- Use outer CV as the main result.
- Treat true-score oracle as a diagnostic upper bound only.
- Use predicted scores for ranking experiments when classification gains are small.
